# QuickPCA on JAX — GPU/TPU acceleration

QuickPCA's compute kernels (Kabsch alignment + SVD-based PCA) sit behind a small
**pluggable backend** interface. The default `numpy` backend runs everywhere; an
optional **`jax`** backend runs the *same* maths through [JAX](https://docs.jax.dev/),
which can JIT-compile and dispatch to a **GPU or TPU** with no change to your
analysis code.

In this notebook we:

1. Discover which backends are registered with `get_backend` / `available_backends`.
2. Run `compute_pca` on `backend="numpy"` and `backend="jax"` and **confirm they agree**.
3. Do a tiny **timing comparison**.
4. Note how GPU/TPU dispatch works.

> The backend selection API is stable regardless of which backends are installed:
> you ask for a backend by name, and QuickPCA tells you clearly if it isn't
> available. This notebook is written to run end-to-end **whether or not** the JAX
> backend is registered in your environment.

> QuickPCA was developed by **Gleb Novikov**. MIT licensed.

## 1. Which backends are available?

`available_backends()` returns the names QuickPCA can hand you, and
`get_backend(name)` returns an instance (raising a clear `ValueError` for unknown
names). The built-in `numpy` backend is always present. The `jax` backend
registers automatically **if** the optional JAX dependency *and* the JAX backend
module are installed (`pip install "quickpca[jax]"`).

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from quickpca import available_backends, get_backend

backends = available_backends()
print(f"registered backends: {backends}")

JAX_BACKEND = "jax" in backends
print(f"JAX backend registered: {JAX_BACKEND}")

# Independently, check whether the JAX library itself is importable and what
# hardware it sees. (The backend can be absent even when jax imports — e.g. on a
# base install where only the numpy backend ships.)
try:
    import jax

    JAX_LIB = True
    print(f"jax library version : {jax.__version__}")
    print(f"jax devices         : {jax.devices()}")
    print(f"default device kind : {jax.devices()[0].platform.upper()}")
except Exception as exc:  # pragma: no cover - environment dependent
    JAX_LIB = False
    print(f"jax library not importable: {exc}")

## 2. Load the data once

We reuse the bundled sample data. The coordinate array feeds every backend
identically, so any difference in results would come purely from the backend's
numerics — which is exactly what we want to test.

In [ ]:
from quickpca import load_trajectory

traj = load_trajectory(
    "../data/reference.pdb",
    "../data/trajectory.nc",
    selection="name CA",
    interval=10,  # stride to keep the demo fast
)
print(f"coords: {traj.coords.shape}  (frames, atoms, xyz)")

## 3. Run NumPy vs JAX and compare

We call `compute_pca(..., backend=...)` once per available backend. If the JAX
backend isn't registered we fall back to running NumPy twice — the comparison
machinery and timing still work, and the notebook still executes cleanly. We make
the situation explicit in the output either way.

A subtlety of PCA: eigenvectors are only defined **up to sign**, and different
linear-algebra libraries may flip a component. QuickPCA's backends apply the same
sklearn-style sign convention, so projections should match directly; we still
compare on **absolute value** as a robust, sign-insensitive check of the
underlying subspace.

In [ ]:
import numpy as np

from quickpca import compute_pca

N_COMPONENTS = 10

# Always run the reference NumPy backend.
pca_np = compute_pca(traj.coords, n_components=N_COMPONENTS, backend="numpy")

# Run JAX if its backend is registered, else re-run numpy as a stand-in so the
# rest of the notebook executes unchanged.
second_backend = "jax" if JAX_BACKEND else "numpy"
pca_other = compute_pca(traj.coords, n_components=N_COMPONENTS, backend=second_backend)

print(f"backend A: {pca_np.backend}")
print(f"backend B: {pca_other.backend}"
      + ("" if JAX_BACKEND else "   (JAX backend not registered — using numpy as a stand-in)"))

In [ ]:
# Compare the two runs. Explained-variance ratios are sign-independent; the
# projections we compare on absolute value to be robust to eigenvector sign flips.
evr_match = np.allclose(
    pca_np.explained_variance_ratio,
    pca_other.explained_variance_ratio,
    atol=1e-6,
)
proj_match = np.allclose(
    np.abs(pca_np.projections),
    np.abs(pca_other.projections),
    atol=1e-5,
)
max_evr_diff = np.max(np.abs(
    pca_np.explained_variance_ratio - pca_other.explained_variance_ratio
))

print(f"explained-variance ratios agree : {evr_match}  (max |Δ| = {max_evr_diff:.2e})")
print(f"projections agree (abs)         : {proj_match}")

if JAX_BACKEND:
    print("\n==> NumPy and JAX produce numerically equivalent PCA results.")
else:
    print("\n==> JAX backend not present here; install it with "
          "`pip install \"quickpca[jax]\"` to run the real cross-backend check.")

## 4. A tiny timing comparison

We time a few repeats of the full `compute_pca` call per backend. A couple of
caveats so the numbers are read honestly:

- **This dataset is tiny.** On a small CA-only trajectory the NumPy backend is
  often *faster* because JAX pays a one-off **JIT-compilation** cost and host↔device
  transfer overhead that only amortises on large problems.
- **Warm-up matters.** The first JAX call compiles; we run an untimed warm-up
  first and `block_until_ready`-style force completion by reading a result value,
  so we measure steady-state execution rather than compilation.

The point of this cell is *methodology*, not to crown a winner on a toy input.

In [ ]:
import time


def timeit_pca(backend: str, repeats: int = 5) -> float:
    # Warm-up (triggers JIT compilation for JAX; reading a value forces execution).
    _ = compute_pca(traj.coords, n_components=N_COMPONENTS, backend=backend)
    _ = float(_.explained_variance_ratio[0])

    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter()
        res = compute_pca(traj.coords, n_components=N_COMPONENTS, backend=backend)
        _ = float(res.explained_variance_ratio[0])  # force completion
        best = min(best, time.perf_counter() - t0)
    return best


t_numpy = timeit_pca("numpy")
print(f"numpy : {t_numpy * 1e3:8.2f} ms / run (best of 5, after warm-up)")

if JAX_BACKEND:
    t_jax = timeit_pca("jax")
    print(f"jax   : {t_jax * 1e3:8.2f} ms / run (best of 5, after warm-up)")
    ratio = t_numpy / t_jax
    faster = "JAX" if ratio > 1 else "NumPy"
    print(f"\n{faster} is {max(ratio, 1 / ratio):.2f}x faster on this (small) input.")
else:
    print("jax   : backend not registered — skipping JAX timing.")
    print("\nOn this tiny CA-only trajectory NumPy is typically the quicker choice;")
    print("JAX's advantage appears on large systems and long trajectories, especially on a GPU/TPU.")

## 5. Using a GPU or TPU

The JAX backend dispatches to whatever device JAX selects by default, so getting
on a GPU/TPU is an **environment** concern, not a code change:

- **GPU:** install a CUDA-enabled jaxlib, e.g.
  `pip install -U "jax[cuda12]"`. JAX then places arrays on the GPU automatically
  and `jax.devices()` reports `cuda` devices.
- **TPU:** on a TPU VM, `pip install -U "jax[tpu]"`; `jax.devices()` reports `tpu`.
- **Pin a device** explicitly if needed:

  ```python
  import jax
  with jax.default_device(jax.devices("gpu")[0]):
      pca = compute_pca(traj.coords, backend="jax")
  ```

Because QuickPCA hides all of this behind `backend="jax"`, the *same analysis
script* scales from your laptop's CPU to a multi-GPU node with no edits — you only
change the backend string (and, for big systems, drop the `interval` stride).

The cell below reports what JAX sees right now.

In [ ]:
if JAX_LIB:
    import jax

    dev = jax.devices()[0]
    print(f"JAX will run on: {dev}  (platform: {dev.platform.upper()})")
    if dev.platform == "cpu":
        print("No accelerator detected — running on CPU. Install jax[cuda12] or "
              "jax[tpu] on suitable hardware to accelerate the 'jax' backend.")
    else:
        print("Accelerator detected — the 'jax' backend will use it automatically.")
else:
    print("JAX is not importable in this environment; only the 'numpy' backend is available.")

## Summary

- QuickPCA's backend interface (`available_backends`, `get_backend`,
  `compute_pca(..., backend=...)`) lets you switch compute engines **without
  touching your analysis code**.
- The `numpy` and `jax` backends compute numerically equivalent PCA results.
- On tiny inputs NumPy wins on raw latency; **JAX's payoff is large systems on
  GPU/TPU**, where it offloads alignment and SVD to the accelerator.

---
*QuickPCA — developed by Gleb Novikov. MIT licensed.*